# Earnings and simple regression with RLMS-HSE — class solutions

Run the cells in order. The code follows the seminar, with short notes on the results. Prepare the 2024 RLMS extract using the [local setup instructions](README.md#rlms-hse-data). Record previews and plots of individual observations can be viewed by running the notebook locally.

## E-01 · Inspect the RLMS-HSE extract

Complete the RLMS download and preparation steps in [README](README.md#rlms-hse-data).
Import pandas, NumPy, Matplotlib, seaborn, and `ols` from `statsmodels.formula.api`.
Read `data/rlms_2024_earnings.csv` into `raw`. Inspect its shape, year values,
first rows, and summary statistics for `earnings_rub`, `hours_week`, and `age`.

This file has one observation per person in 2024. `earnings_rub` comes from J10:
after-tax money received from the main job in the last 30 days, in nominal rubles.
It is not hourly pay, total household income, or an annual salary.
`hours_week` is usual weekly hours at the main job, not hours in those same 30 days.
Check variable definitions and response codes in [data/README.md](data/README.md).

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import statsmodels
from statsmodels.formula.api import ols

sns.set_theme(style="whitegrid", context="notebook")

In [3]:
raw = pd.read_csv("data/rlms_2024_earnings.csv")
raw.head()

In [4]:
raw.shape

(16882, 8)

In [5]:
raw["year"].unique()

array([2024.])

In [6]:
raw["earnings_rub"].isna().sum() / len(raw["earnings_rub"])

np.float64(0.5508233621608815)

In [7]:
raw[["earnings_rub", "hours_week", "age"]].describe()

Each record represents one respondent in the 2024 extract. Earnings measure after-tax receipts from the main job in the last 30 days, in rubles. The initial numerical summary still includes special response codes.

## E-02 · Make the analysis sample explicit

Create `clean` by replacing special codes 99999995 through 99999999 with `np.nan`.
These include nonresponse codes; for usual hours, 99999995/99999996 represent
other/irregular arrangements and do not give a numerical weekly total.

Create `selected`: age 18–65 inclusive; currently working (`work_status == 1`);
`sex_code` in [1, 2]; observed strictly positive earnings.
Create `work`: keep usual weekly hours between 1 and 168 inclusive.
Report the number of rows at each step and the number excluded for hours.
Do not replace missing earnings with zero or remove high earnings automatically.

In `work`, create `female` (1 for code 2; 0 for code 1) and `sex` (Male/Female).
These names reflect the two categories in the source variable, not a complete
measure of gender identity. Both main regressions must use this same `work` table.

Useful tools: `.replace()`, `.between()`, `.isin()`, `.gt()`, `.copy()`, `.astype(int)`, `.map()`.

In [8]:
missing_codes = [99999995, 99999996, 99999997, 99999998, 99999999]
clean = raw.replace(missing_codes, np.nan)
clean.head()

In [9]:
selected = clean.loc[clean["age"].between(18, 65) & clean["work_status"].eq(1) & clean["sex_code"].isin([1, 2]) & clean["earnings_rub"].gt(0)]
selected.head()

In [10]:
selected.shape

(6981, 8)

In [11]:
work = selected.loc[selected["hours_week"].between(1, 168)].copy()
work.head()

In [12]:
work.shape

(6239, 8)

In [13]:
work["female"] = work["sex_code"].eq(2).astype(int)
work.head()

In [14]:
work["sex"] = work["sex_code"].map({1: "Male", 2: "Female"})
work.head()

In [15]:
print("original rows: ", len(raw))
print("Working adults with positive earnings: ", len(selected))
print("Analysis sample with observed usual hours: ", len(work))
print("Excluded for missing or invalid hours: ", len(selected) - len(work))

original rows:  16882
Working adults with positive earnings:  6981
Analysis sample with observed usual hours:  6239
Excluded for missing or invalid hours:  742


The age, work status, recorded sex and earnings conditions select 6,981 respondents. Requiring valid usual weekly hours excludes another 742, leaving 6,239 respondents for both regressions.

## E-03 · Compare means, medians, and sample sizes

Group `work` by `sex` and summarize earnings with count, mean, and median.
Retrieve male and female mean earnings from this table.
Define the descriptive earnings gap as:

`(male mean − female mean) / male mean × 100`.

Print the absolute difference in rubles and the gap in percent. State the denominator.
Explain why a mean and a median can tell different stories and why this gap does
not by itself establish discrimination or an hourly-pay difference.

In [16]:
summary = work.groupby("sex")["earnings_rub"].agg(["count", "mean", "median"])
summary

,count,mean,median
sex,,,
Female,3375,43001.979917,36000.0
Male,2864,61068.412709,50000.0


In [17]:
male_mean = summary.loc["Male", "mean"]
female_mean = summary.loc["Female", "mean"]
print(male_mean, female_mean)

61068.41270949721 43001.97991703704


In [18]:
gap_rub = male_mean - female_mean
gap_pct = gap_rub / male_mean * 100
print(gap_rub, gap_pct)

18066.432792460168 29.583923981129644


Mean receipts were about 61,068 rubles for men and 43,002 rubles for women in the selected sample. The difference was about 18,066 rubles, or 29.6% of the male mean. A long upper tail can raise the mean above the median. The measure describes monthly receipts and combines differences in hours, jobs and other characteristics.

## E-04 · Visualize earnings distributions

Create a histogram of positive earnings in `work`, using 40 bins.
Label earnings in rubles and the vertical axis as respondents.
Then create a boxplot of earnings by `sex`, keeping high observations visible.
Use a logarithmic y-axis on the boxplot so the long right tail is readable.

Explain what a bin represents. Explain the box, median line, whiskers,
and points beyond the whiskers. On the log axis, equal distances represent
equal ratios, not equal differences in rubles. Do not interpret bar lengths
or vertical distances as though the scale were linear.

In [19]:
work.head()

In [20]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(data=work, x="earnings_rub", bins=40)
ax.set(title="Earnings distribution", xlabel="Main-job earnings in last 30 days in RUB", ylabel="Respondents")
fig.tight_layout()
plt.show()

In [21]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=work, x="sex", y="earnings_rub", order=["Male", "Female"], ax=ax)
ax.set_yscale("log")
ax.set(title="Earnings by sex box-plot", xlabel="Sex", ylabel="Earnings in RUB (logarithmic scale)")
fig.tight_layout()
plt.show()

The histogram counts respondents in each earnings interval. The box spans the middle half of the sample, with a line at its median. Whiskers extend to the most extreme observations within 1.5 interquartile ranges of the box. More distant observations appear separately. Equal vertical steps on the log axis represent equal earnings ratios.

## E-05 · A simple regression reproduces a group comparison

Fit OLS with formula `earnings_rub ~ female` on `work`. Save the fitted model as `m0`.
Use `.fit(cov_type="HC3")` for heteroskedasticity-robust standard errors.
Display the coefficients, confidence intervals, number of observations, and R-squared.

Interpret the intercept and the coefficient on `female` in rubles.
Check the coefficient against female mean minus male mean from E-03.
Why does its sign differ from the gap definition in E-03?

Formula notation: the outcome is left of `~`; predictors are right of it.
An intercept is included by default. HC3 does not account for the survey's
clustering/weights and does not make this a causal or nationally representative estimate.

In [22]:
work.head()

In [23]:
m0 = ols("earnings_rub ~ female", data=work).fit(cov_type="HC3")

In [24]:
m0.params.to_frame("Coefficient (RUB)")

,Coefficient (RUB)
Intercept,61068.412709
female,-18066.432792


In [25]:
m0.params["Intercept"] + m0.params["female"]

np.float64(43001.97991703701)

In [26]:
m0.conf_int().rename(columns={0: "Lower 95%", 1: "Upper 95%"})

,Lower 95%,Upper 95%
Intercept,59542.399838,62594.425581
female,-19859.535875,-16273.329710


In [27]:
m0.nobs

6239.0

In [28]:
m0.rsquared

np.float64(0.062416668576836365)

The intercept equals male mean earnings. The female coefficient equals female minus male mean earnings, about −18,066 rubles. E-03 subtracts in the opposite order.

## E-06 · Add age and usual weekly hours

Fit `earnings_rub ~ female + age + hours_week` on the same `work` table as E-05,
again with HC3 standard errors. Save the result as `m1`.
Compare the coefficients, sample sizes, and R-squared of both models.

Interpret the units of the age and hours coefficients and the adjusted female
coefficient. Distinguish a conditional association from a causal effect.
Name two omitted factors and one possible problem with a linear age term.
Do not interpret the intercept as typical earnings for a worker aged zero.

In [29]:
m1 = ols("earnings_rub ~ female + age + hours_week", data=work).fit(cov_type="HC3")

comparison = pd.concat([m0.params.rename("Group-only model"), m1.params.rename("Age + hours model")], axis=1)
comparison

,Group-only model,Age + hours model
Intercept,61068.412709,55277.253099
female,-18066.432792,-16346.460730
age,NaN,-220.760182
hours_week,NaN,336.292309


In [30]:
m0.conf_int().rename(columns={0: "Lower 95%", 1: "Upper 95%"})

,Lower 95%,Upper 95%
Intercept,59542.399838,62594.425581
female,-19859.535875,-16273.329710


In [31]:
m1.conf_int().rename(columns={0: "Lower 95%", 1: "Upper 95%"})

,Lower 95%,Upper 95%
Intercept,50174.032728,60380.473470
female,-18141.246558,-14551.674903
age,-287.953249,-153.567115
hours_week,247.512378,425.072240


In [32]:
m1.nobs

6239.0

In [33]:
m1.rsquared

np.float64(0.07664861029492398)

At the same age and usual weekly hours, fitted monthly receipts for women are about 16,346 rubles lower. The age slope is about −221 rubles per additional year and the hours slope about +336 rubles per additional usual weekly hour, holding the other predictors fixed. Industry and occupation are omitted, and a single age slope cannot capture a mid-career peak.

## E-07 · Predictions and residuals

Create two hypothetical profiles with age 35 and usual weekly hours 40;
set `female` to 0 and 1. Use `m1.predict()` to obtain fitted earnings.
Explain why these are fitted values, not guarantees or treatment effects.

Plot residuals against fitted values for the observed sample, with transparency
and a horizontal reference line at zero. A residual is observed minus fitted earnings.
Describe one feature that makes a simple linear model incomplete.
These are in-sample diagnostics, not a test of predictive performance on new data.

In [34]:
profiles = pd.DataFrame({"female": [0, 1], "age": [35, 35], "hours_week": [40, 40]})

In [35]:
m1.predict(profiles)

0    61002.339108
1    44655.878378
dtype: float64

In [36]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(m1.fittedvalues, m1.resid, alpha=0.15, s=12)
ax.axhline(0, color="black", linestyle="--")
ax.set(xlabel="Fitted earnings (RUB)", ylabel="Residual (RUB)")
fig.tight_layout()
plt.show()

For the two profiles, the predicted difference equals the female coefficient because age and weekly hours are held fixed. Predictions describe model averages. The residuals show how far observed receipts can depart from those averages, particularly for high earners.

## E-08 · Check sensitivity and write a cautious conclusion

Define `prime_age` as the subset of `work` aged 25–54 inclusive.
Refit the age + hours model, using the same formula and HC3 covariance.
Compare the female coefficient and sample size with `m1`.
This changes the population being described; it is not a search for the most
favourable result and does not establish causality.

Write a short result paragraph in your own words with:
the dataset/year; the sample and earnings definition; the descriptive gap;
the adjusted coefficient with units; and at least two limitations.
Explain what changed in the sensitivity check. Save the notebook.

In [37]:
prime_age = work.loc[work["age"].between(25, 54)]
m_prime = ols("earnings_rub ~ female + age + hours_week", data=prime_age).fit(cov_type="HC3")

In [38]:
sensitivity = pd.DataFrame(
    {
        "Sample": ["Ages 18-65", "Ages 25-54"],
        "Observations": [m1.nobs, m_prime.nobs],
        "Female coefficient (RUB)": [m1.params["female"], m_prime.params["female"]]
    }
)

sensitivity

,Sample,Observations,Female coefficient (RUB)
0,Ages 18-65,6239.0,-16346.460730
1,Ages 25-54,4741.0,-18344.114718


For 6,239 working respondents aged 18–65 in RLMS-HSE 2024 with positive main-job receipts and valid hours, the unweighted mean gap was 29.6% of male mean receipts. The adjusted female coefficient was about −16,346 rubles for the last 30 days. Restricting the analysis to ages 25–54 left 4,741 respondents and changed the coefficient to about −18,344 rubles. The sample excludes workers with irregular or missing hours, and the models omit occupation and industry.

## E-09 · Independent practice · education categories and log earnings

Use only `work` rows with `education_code` in 1–6, and create `log_earnings = np.log(earnings_rub)`.
Fit `log_earnings ~ female + age + hours_week + C(education_code)`.
`education_code` is a category, **not years of schooling**. `C(...)` requests
separate category effects, with one omitted reference category.

Convert the female coefficient `b` to `100 × (np.exp(b) − 1)`.
Interpret this as a model-implied conditional ratio of geometric means, not
automatically a percentage difference in arithmetic mean earnings or a causal effect.
Compare sample sizes before comparing coefficients. Log transformation changes
the outcome and the estimand, so its coefficients are not ruble differences.

This optional extension was left for independent practice.